<font size="10">**Business Case: ML predittivo e Large Language Models**</font><br>

> (c) 2026 Antonio Michele Piemontese

**Business Case — Early Warning per l'abbandono / retention della Clientela**

Machine Learning (classificazione) + accesso a un LLM via API

Dataset di riferimento: `clienti_banca_churn.csv`


# Legenda delle icone (standard)
Legenda delle icone (standard) usate nel notebook (per garantire consistenza semantica):

👉 punto di attenzione, il "succo"<br>
📌 nota / inizio di una nota<br>
📦 punto elenco importante<br>
📊 dati/numeri<br>
🔹 punto elenco normale<br>
⭐ punto elenco importante<br>
✅ punto risolto, positivo<br>
❌ punto negativo, da evitare<br>
⚠️ attenzione, warning, allarme<br>
💡 idea chiave<br>
🧠 idea intuitiva<br>
📝 sintesi / bottom-line<br>
⟶ conseguenza, effetto, passo successivo

# Immagini

* `ROC-AUC1.png`
* `ROC-AUC2.png`
* `ROC-AUC3.png`

# Rilevamento ambiente di sviluppo

In [ ]:
# impostazione del TOGGLE BINARIO:
try:
    import google.colab                      # package disponibile SOLO in Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running on Colab:", IN_COLAB)


# IMPORT dei package necessari (necessari sia in JN che in Colab):
from IPython.display import Image, display   # import dei package di incorporamento e visualizzazione immagine (una tantum)
                                             # Image e display sono entrambi necessari a Jupyter Notebook
                                             # Google Colab utilizza solo Image
import os                                    # necessario a Google Colab per vedere da una cella codice
                                             # i contenuti del 'content'



# Contesto e obiettivo di business

Una grande banca retail europea osserva che, ogni anno, una quota della clientela chiude il rapporto e sposta liquidità e patrimonio verso concorrenti tradizionali, banche online e fintech. Acquisire un nuovo cliente costa molto più che trattenerne uno esistente; inoltre, la perdita di un cliente affluent significa anche perdita di patrimonio in gestione (AUM) e di ricavi commissionali ricorrenti.

Il problema è che oggi i gestori se ne accorgono quando l’abbandono è già in corso: il cliente ha smesso di operare, ha chiesto informazioni sulla chiusura, ha spostato i bonifici altrove. L’intervento arriva tardi, quando ormai c’è poco da fare.

> **Obiettivo**
>
> Passare da un approccio reattivo a uno proattivo: identificare in anticipo i clienti a maggior rischio di abbandono, in particolare se clienti con AUM medio-alto, capire perché sono a rischio e mettere il gestore nelle condizioni di agire con un’azione mirata (*retention action*), prima che sia troppo tardi.


# Il problema in dettaglio

Il problema si scompone in due domande operative:

- **CHI è a rischio?** → un problema di **classificazione binaria**: per ogni cliente, stimare la probabilità di abbandono nei mesi successivi (1, 3 o 6).
- **COSA fare?** → per i clienti segnalati (rischio di abbandono), produrre (a) una **spiegazione** comprensibile del perché sono a rischio e (b) una **proposta di azione di retention** <u>personalizzata</u>, pronta per il gestore.

La prima domanda è il terreno classico del machine learning supervisionato. La seconda richiede di trasformare numeri e probabilità in linguaggio e decisioni: è qui che entra in gioco l’LLM.


# Perché un approccio ibrido (ML + LLM)

Né il solo modello di ML né il solo LLM risolvono il problema da soli:

- **Un modello di classificazione** è ottimo per stimare il rischio su migliaia di clienti in modo coerente e misurabile, ma restituisce un numero (es. 0,73) e un elenco di feature: poco utilizzabile da un gestore non tecnico.
- **Un LLM** è ottimo per spiegare, riassumere e redigere testi, ma non è uno strumento affidabile per stimare una probabilità di abbandono da dati tabellari: tende a essere incoerente, non calibrato, e non si misura con le metriche statistiche abituali.

La combinazione sfrutta i punti di forza di entrambi: il modello fa la previsione (il « cervello »), l’LLM la traduce in spiegazione e azione (la « voce »).

> **Nota per gli analisti di business**
>
> L’LLM non « spiega » il modello. La spiegazione delle ragioni del rischio nasce dai metodi di interpretabilità (importanza delle feature, valori SHAP, o semplicemente i valori delle feature del cliente). L’LLM si limita a rendere quel risultato in linguaggio naturale: la fonte della spiegazione è statistica, non l’LLM.


# Installazione package

Il package `shap` per il calcolo dei [valori di Shapley](https://it.wikipedia.org/wiki/Valore_di_Shapley) non fa parte dei circa 650 pre-installati in Google Colab a giugno 2026.

In [ ]:
!pip show shap  # --> non installato?!

: 

In [6]:
import sys;print(sys.executable);print(sys.version)

c:\Users\Utente\AppData\Local\spyder-6\python.exe
3.11.13 | packaged by conda-forge | (main, Jun  4 2025, 14:39:58) [MSC v.1943 64 bit (AMD64)]


Nel caso il package `shap` non sia tra quelli pre-installati in Google colab lo dobbiamo installare (sempre sperando di non avere conflitti sulle versioni).

In [ ]:
# 7-8" circa
!pip install shap

: 

"*Requirement already satisfied*" significa semplicemente che i sotto-package necessari a `shap` sono già installati.

#Import dei package

Per prassi di buona programmazione python le import dei vari package sono messe tutte all'inizio del notebook, anzichè al momento in cui servono.

Per il ML predittivo useremo la famosa libreria Python [scikit-learn](https://scikit-learn.org/stable/).

In [ ]:
# circa 6-20"

import pandas as pd               # gestione dataframe
import numpy as np                # gestione array e calcoli numerici
import matplotlib.pyplot as plt   # il principale package per fare plot (diagramma)

# pipeline minimale di classificazione (scikit-learn)
from sklearn.ensemble import RandomForestClassifier    # il classificatore binario
from sklearn.model_selection import train_test_split   # per dividere il dataset in due parti: training e test

# metriche tecniche (NON per il gestore) di valutazione della classificazione: curva ROC e AU
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.metrics import recall_score, precision_score, f1_score
from sklearn.metrics import classification_report

# shap (per valori di Shapley)
import shap                         # prima fare 'pip install shap' da terminale

from google.colab import userdata   # per usare le secret di Google Colab

from openai import OpenAI           # per usare gli LLM OpenAI  (da NON confondere con chatGPT)

# moduli della libreria standard di Python (ci sono già, non vanno installati)
import textwrap                     # per formattare un testo
import json                         # per lavorare con i dati in formato JSON (i modelli LLM dialogano con la app python in JSON)

# Il dataset

👉 Il dataset fornito è **sintetico** (cioè, creato *ad hoc*), ma costruito con **relazioni realistiche tra comportamento del cliente e abbandono**. Simula i dati che una banca otterrebbe integrando il core banking (saldi, prodotti), il CRM (anzianità, segmento, note del gestore), i canali digitali (accessi all’app, operatività) e i sondaggi di soddisfazione.

Caricamento in pandas: `df = pd.read_csv("clienti_banca_churn.csv")`

### In sintesi

- **4.200 clienti**, 20 colonne: 1 identificativo, 18 feature (di cui una testuale) e 1 target.
- **Tasso di abbandono ≈ 14%**: un problema sbilanciato e realistico (la maggior parte dei clienti resta).
- Abbandono per segmento: mass 15,8%, private 11,0%, affluent 9,3%. Il segmento mass è il più volatile, ma **l’abbandono di un cliente affluent/private « pesa » molto di più** in termini di patrimonio.
- Note del gestore presenti per **1.489 clienti (35,5%)**: testo libero, input per il modulo LLM.
- Circa l’8% di valori mancanti nella colonna `nps`: occasione per discutere la gestione dei missing.
- Outlier: pochi e identificabili — col dataset ripulito ogni variabile presenta solo lo 0–4% di valori anomali; i casi che contano sono i ~40 clienti "facoltosi ma inattivi" isolati dall'analisi bivariata (abbandono 36% contro 14%) e i pochi outlier multivariati in cima alla distanza di Mahalanobis, da segnalare e capire, non rimuovere.
- Saldo di conto: valore **mediano ≈ 6.850 €** (la media, ≈ 26.000 €, è molto più alta: la distribuzione è asimmetrica, con pochi clienti dai saldi elevati). AUM mediano tra gli investitori ≈ 41.400 €.


In [ ]:
df = pd.read_csv("clienti_banca_churn.csv")   # senza il parametro dtype (delle singole colonne)
df.head()

## Dizionario dei dati

Da 4 fonti, come detto:
* core banking
* CRM
* canali
* sondaggi di soddisfazione (*customer satisfaction*)

| Campo | Tipo | Descrizione |
| --- | --- | --- |
| `id_cliente` | testo | Identificativo univoco del cliente (non usato come feature). |
| `eta` | intero | Età del cliente in anni. |
| `segmento` | categoriale | Segmento commerciale: mass, affluent, private. |
| `anzianita_mesi` | intero | Durata della relazione con la banca, in mesi. |
| `num_prodotti` | intero | Numero di prodotti sottoscritti (conto, carte, mutuo, fondi…). |
| `stipendio_accreditato` | binario | 1 se stipendio o pensione viene accreditato sul conto. |
| `ha_carta_credito` | binario | 1 se possiede una carta di credito. |
| `ha_mutuo` | binario | 1 se ha un mutuo in essere. |
| `ha_investimenti` | binario | 1 se detiene prodotti di investimento. |
| `saldo_medio_eur` | decimale | Saldo medio di conto corrente (EUR). **[Monetary — valore]** |
| `patrimonio_gestito_eur` | decimale | Patrimonio investito / AUM (EUR); 0 se non investe. **[Monetary — valore]** |
| `canale_preferito` | categoriale | Canale più usato: app, web, filiale. |
| `accessi_app_mese` | intero | Accessi medi all’app mobile al mese. **[Frequency — engagement]** |
| `transazioni_mese` | intero | Numero medio di operazioni al mese. **[Frequency]** |
| `giorni_ultima_operazione` | intero | Giorni dall’ultima operazione. **[Recency]** |
| `variazione_saldo_3m_pct` | decimale | Variazione % del saldo negli ultimi 3 mesi (negativa = in calo). |
| `reclami_12m` | intero | Numero di reclami negli ultimi 12 mesi. |
| `nps` | decimale | Indice di soddisfazione 0–10 (può essere mancante). |
| `note_gestore` | testo | Nota libera del gestore (≈35% dei clienti). Input per l’LLM. |
| `abbandono` | **binario (TARGET)** | 1 se il cliente ha lasciato la banca nella finestra di osservazione. |

*Tabella 1 — Dizionario completo del dataset clienti_banca_churn.csv*

> **Una chiave di lettura: il framework RFM**
>
> Cinque colonne si leggono come il classico schema **RFM** del marketing:  **Recency** (giorni_ultima_operazione, da quanto il cliente non opera), **Frequency** (transazioni_mese, con accessi_app_mese come engagement digitale) e **Monetary**.
> Un punto a cui prestare attenzione: nel RFM retail il « Monetary » è quanto il cliente _spende_. Qui quel dato non c’è (transazioni_mese è un conteggio, non un importo): la versione bancaria del Monetary è quanto il cliente _vale_, cioè il suo patrimonio presso l’istituto — saldo_medio_eur più patrimonio_gestito_eur. È una misura di valore (stock), non di spesa (flusso): lo stesso « valore » che useremo per pesare le priorità di retention.

## Cosa distingue chi resta da chi abbandona

Già **a livello descrittivo** emergono differenze coerenti tra le due popolazioni: sono i segnali che il modello imparerà a combinare.

| Indicatore (media) | Clienti che restano | Clienti che abbandonano |
| --- | --- | --- |
| Accessi app / mese | 8,2 | 5,9 |
| Transazioni / mese | 12,4 | 9,4 |
| Giorni dall’ultima operazione | 12,4 | 14,7 |
| Reclami (12 mesi) | 0,30 | 0,57 |
| Soddisfazione (NPS 0–10) | 7,1 | 6,5 |
| Numero di prodotti | 3,2 | 2,6 |
| Stipendio accreditato (quota) | 62% | 30% |
| Anzianità (mesi) | 121 | 103 |

*Tabella 2 — Profilo medio: chi abbandona è meno attivo, più insoddisfatto, con meno prodotti e relazione più recente.*


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
print(df['abbandono'].value_counts())  # conteggi assoluti
print(df['abbandono'].value_counts()[0] / len(df))
print(df['abbandono'].value_counts()[1] / len(df))

14% di abbandoni è un valore ragionevole. Ci sta come dinamica e permette di "imparare anche gli abbandoni".

Il dataset di training deve essere "bilanciato", cioè con abbastanza righe per entrambe le classi in modo che il modello impari a riconoscere le relazioni tra i predittori/feature e la risposta **per entrambe le classi**.

Analisi NA (Not Available) / MV (Missing values)

In [ ]:
df.isna().sum()

Contraddizione con la `df.info`.

Due dei possibili modi per gestire i dati mancanti è quello di:
* cancellare ogni riga che abbia almeno una cella MV --> rischio: eliminare molte righe (ma per allenare bene un modello predittivo serve una storia passata la più ampia possibile).

In [ ]:
df["nps"].median()  # più affidabile della media

In [ ]:
df["nps"] = df["nps"].fillna(df["nps"].median())

In [ ]:
df.isna().sum()

**Gestione degli outlier**

Nel file csv attuale (`clienti_banca_churn`) - per come è stato creato - quasi tutte le colonne sono ben delimitate (età 20–85, transazioni 1–30, prodotti 1–7…) e non producono outlier. C'è invece una "valanga" di outlier che viene praticamente solo dalle **due colonne monetarie con code lognormali estreme**.<br>

> saldo e patrimonio sono lognormali (σ≈0,9–1,0) moltiplicate per un fattore di segmento 1/6/30, e quella miscela moltiplicativa fa esplodere la coda (i private finiscono a milioni).

Utilizziamo quindi, per il rilevamento degli outlier, **un altro file csv**, identico in tutto il resto (segnale churn, profilo, note, mancanti NPS), ma con ~20 outlier controllati di tipi diversi iniettati.

In questo secondo file *csv*, di nome `clienti_churn_banca_outlier`, distribuzioni monetarie moderate e **~18 outlier controllati iniettati** di tipi diversi (univariati estremi, combinazioni "impossibili", clienti ricchi-dormienti). Serve a mostrare che i metodi trovano *pochi* casi netti, non un terzo della base. Il segnale di churn e la tabella profilo del cap. 4 restano invariati (ROC-AUC ≈ 0,76).

**1. Univariata — IQR e z-robusto (mediana/MAD)**

Punto chiave: outlier statistico ≠ errore. Guardate cosa esce sul saldo:

In [ ]:
# definizione delle due funzioni preparatorie:
def flag_iqr(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75); iqr = q3 - q1
    return (s < q1 - k*iqr) | (s > q3 + k*iqr)

def flag_mad(s, thr=3.5):                      # z robusto: usa mediana e MAD
    med = s.median(); mad = (s - med).abs().median()
    if mad == 0: return pd.Series(False, index=s.index)
    return (0.6745 * (s - med) / mad).abs() > thr


Entrambe le funzioni sono **rilevatori di outlier univariati** (lavorano su una colonna alla volta) e restituiscono una **maschera booleana** — una Series di `True`/`False`, stesso indice dell'input, con `True` dove il valore è considerato anomalo. Per questo poi puoi fare `.sum()` per contarli o usarle per indicizzare il DataFrame.

**`flag_iqr` — regola di Tukey.** Calcola il primo e terzo quartile (`q1`, `q3`) e la loro distanza, l'IQR (l'ampiezza del 50% centrale dei dati). Poi traccia due "recinti": `q1 − 1.5·IQR` sotto e `q3 + 1.5·IQR` sopra. Tutto ciò che cade fuori è flaggato. Il moltiplicatore `k=1.5` è la soglia classica; con `k=3` parleresti di outlier "estremi". Essendo basata su quartili, è di per sé robusta: non usa media né deviazione standard, quindi non viene distorta dai valori estremi che sta cercando.

**`flag_mad` — z-score robusto.** È l'analogo robusto del classico z-score. Lo z-score normale è `(x − media)/std`, ma media e std sono entrambe gonfiate proprio dagli outlier, quindi il metodo "si autoacceca". Qui invece si usano **mediana** al posto della media e **MAD** (Median Absolute Deviation = `median(|x − mediana|)`) al posto della std, due stimatori che resistono agli estremi. Il punteggio diventa `0.6745·(x − mediana)/MAD` e si flagga quando il suo valore assoluto supera `thr=3.5`.

Il pezzo non ovvio è quel **`0.6745`**: serve a rendere il MAD confrontabile con una deviazione standard. Per una distribuzione normale, infatti, `MAD ≈ 0.6745·σ` (0.6745 è il quantile 0.75 della normale standard). Moltiplicando per quel fattore, il denominatore diventa una stima robusta di σ, e così la soglia `3.5` si legge "≈ 3,5 deviazioni standard" in versione robusta (è il valore raccomandato da Iglewicz & Hoaglin).

Infine la riga di guardia `if mad == 0: return ... False`: se più della metà dei valori è identica, la MAD vale 0 e divideresti per zero. In quel caso la funzione non flagga nulla — ed è esattamente ciò che succede sul `patrimonio_gestito_eur`, dove il 63% di clienti ad AUM = 0 schiaccia mediana e MAD a zero (il motivo per cui là il MAD restituisce 0 outlier mentre l'IQR ne trova il 17%).

Per prima cosa dobbiamo leggere il nuovo file csv con pochi outlier:

In [ ]:
df = pd.read_csv("clienti_banca_churn_outlier.csv")   # senza il parametro dtype (delle singole colonne)
df.head()

La seguente cella stampa la tabella per colonna (da cui si legge **saldo_medio_eur → IQR 532 / 12,7%**, e **patrimonio_gestito_eur → MAD 0**).

E queste due chiamate producono le note di prima.

In [ ]:
# lista python (sottoinsieme) delle colonni sulle quali vogliamo trovare gli outlier:
num_cols = ["eta", "anzianita_mesi", "num_prodotti", "saldo_medio_eur", "patrimonio_gestito_eur",
            "transazioni_mese", "accessi_app_mese", "giorni_ultima_operazione",
            "variazione_saldo_3m_pct", "nps", "reclami_12m"]

# costruzione df di riepilogo
riepilogo = pd.DataFrame({
    "IQR_n": {c: flag_iqr(df[c]).sum() for c in num_cols},   # vettorizzazione per riga
    "MAD_n": {c: flag_mad(df[c]).sum() for c in num_cols},
})
riepilogo["IQR_%"] = (riepilogo["IQR_n"] / len(df) * 100).round(1)
riepilogo["MAD_%"] = (riepilogo["MAD_n"] / len(df) * 100).round(1)
print(riepilogo.sort_values("IQR_n", ascending=False))

4 metriche.

Sono la **stessa informazione espressa in due modi** — conteggio assoluto e percentuale — per ciascuno dei due metodi. Il suffisso `_n` sta per *numero* (quanti record), `_%` per *percentuale* sul totale.

In dettaglio, riga per riga della tabella (dove ogni riga è una colonna numerica del dataset):

- **`IQR_n`** — quanti record ha flaggato come outlier il metodo IQR su quella colonna. Viene da `flag_iqr(df[c]).sum()`: la funzione restituisce una maschera di `True`/`False`, e `.sum()` conta i `True` (in Python `True` vale 1), quindi è il numero di anomalie.
- **`MAD_n`** — lo stesso conteggio, ma col metodo MAD (z-score robusto).
- **`IQR_%`** — `IQR_n` diviso il numero totale di righe (`len(df)` = 4.200) × 100: la quota di clienti flaggati dall'IQR, in percentuale.
- **`MAD_%`** — idem per il MAD.

Concretamente, sulla riga `saldo_medio_eur` leggiamo `IQR_n = 532` e `IQR_%= 12,7`: vuol dire 532 clienti su 4.200, cioè il 12,7%. Le due metriche `_n` ci danno il dato grezzo, le due `_%` lo rendono confrontabile tra colonne (e leggibile a colpo d'occhio: "il 12% dei saldi è fuori soglia").

Il motivo per cui ci sono **due metodi affiancati** (IQR e MAD) è proprio il confronto: dove i due numeri divergono molto — come su `patrimonio_gestito_eur`, IQR al 17% e MAD a 0 — si capisce che la scelta del metodo cambia radicalmente il risultato, ed è lì che sta la lezione.

---
**Risultato sul dataset pulito.** Ogni colonna ha ora pochissimi outlier univariati: `saldo_medio_eur` 3,6%, `anzianita_mesi` 3,3%, `giorni_ultima_operazione` 2,9%, `patrimonio_gestito_eur` 1,6%, tutte le altre sotto l'1% (`eta`, `num_prodotti`, `accessi_app_mese` esattamente 0). Sui monetari prima eravamo al 13–17%: la coda patologica è sparita. Nota che la MAD ora funziona anche sul patrimonio, perché gli zeri sono scesi sotto il 50% (≈46%) e mediana/MAD non collassano più a 0.

---

In [ ]:
# affondo: gli "outlier alti" del saldo sono ricchi legittimi o sono a rischio abbandono?
alti = flag_iqr(df["saldo_medio_eur"])
print(f"Saldo outlier (IQR): {alti.sum()}")
print(f"  abbandono: {df.loc[alti,'abbandono'].mean()*100:.1f}%  (media {df['abbandono'].mean()*100:.1f}%)")

I ~151 clienti con saldo "fuori soglia" abbandonano **meno** della media (13,2% vs 14,0%): sono per lo più clienti facoltosi reali (più i 2 estremi iniettati), non errori e non in fuga. La regola resta **flag, non drop**.

**2. Bivariata — anomalie nella combinazione**

Un cliente può essere "normale" su ogni variabile (cioè dentro + $Q3+IQR * 1.5$ e $Q1-IQR * 1.5$) ma raro nella *coppia*. Caso interessante per la retention: **saldo alto + poche transazioni** (ricco ma disingaggiato).


In [ ]:
import matplotlib.pyplot as plt

# grafico matplotlib (la sua sintassi)
x, y = "transazioni_mese", "saldo_medio_eur"
combo = (df[y] > df[y].quantile(0.90)) & (df[x] <= df[x].quantile(0.10))

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df[x], df[y], s=8, alpha=0.25)
ax.scatter(df.loc[combo, x], df.loc[combo, y], s=30, color="crimson",
           label=f"saldo alto + poche txn ({combo.sum()})")
ax.set_yscale("log"); ax.set_xlabel(x); ax.set_ylabel(y); ax.legend()
ax.set_title("Outlier bivariato: anomalo nella combinazione")
plt.show()

print(f"combo: {combo.sum()} clienti | abbandono {df.loc[combo,'abbandono'].mean()*100:.1f}%")

**Risultato.** Il combo isola **39 clienti** che abbandonano al **35,9%** — due volte e mezza la media — e tra loro ci sono i 4 "ricchi-dormienti" iniettati apposta (saldo 120–180k €, 0–1 transazioni, ~160 giorni di inattività). Il filtro univariato sul saldo da solo li avrebbe persi nel mucchio dei facoltosi fedeli; la combinazione li fa emergere come lista azionabile.

Interpretazione del grafico

Ogni punto è un cliente: in ascissa le transazioni al mese, in ordinata il saldo medio (in scala **logaritmica** — necessaria perché il saldo è asimmetrico all'estremo, skew ≈ 13: senza log i pochi straricchi schiaccerebbero tutti gli altri su una riga piatta in basso).

**3. Mahalanobis + chi-quadro — classica vs robusta**

Approccio multivariato. Prima la versione classica (*as-is*), poi quella robusta (*idiomatica*) affiancata.

In [ ]:
from scipy.stats import chi2
from sklearn.covariance import MinCovDet

mcols = ["eta","anzianita_mesi","num_prodotti","saldo_medio_eur","transazioni_mese",
         "accessi_app_mese","giorni_ultima_operazione","variazione_saldo_3m_pct","nps"]
X = df[mcols].to_numpy(float)
p = X.shape[1]
soglia = chi2.ppf(0.975, df=p)        # sotto normalità multivariata, D² ~ chi²(p)

# as-is: classica (media + covarianza campionaria)
mu = X.mean(axis=0)
d2_classica = np.einsum("ij,jk,ik->i", X-mu, np.linalg.inv(np.cov(X, rowvar=False)), X-mu)

# idiomatica: robusta (MCD, stimatori resistenti agli outlier)
# d2_robusta = MinCovDet(random_state=7).fit(X).mahalanobis(X)

In [ ]:
# fl_cl, fl_rob = d2_classica > soglia, d2_robusta > soglia
fl_cl = d2_classica > soglia
print(f"Soglia chi² (0.975, p={p}) = {soglia:.1f}\n")
print(f"Outlier - Mahalanobis CLASSICA : {fl_cl.sum():4d}  ({fl_cl.mean()*100:.1f}%)")
# print(f"Outlier - Mahalanobis ROBUSTA  : {fl_rob.sum():4d}  ({fl_rob.mean()*100:.1f}%)")
# print(f"  in comune {(fl_cl&fl_rob).sum()} - solo robusta {(fl_rob & ~fl_cl).sum()} (mascheramento)\n")

# top = (df.assign(D2=d2_robusta.round(0))
#         .loc[fl_rob, ["id_cliente","segmento","saldo_medio_eur","transazioni_mese",
#                       "giorni_ultima_operazione","abbandono","D2"]]
#         .sort_values("D2", ascending=False).head(5))
# print("Top 5 outlier (distanza robusta):")
# print(top.to_string(index=False))

**Risultato.** Classica **116 (2,8%)**, robusta **382 (9,1%)** — il mascheramento resta visibile (266 punti emergono solo con la robusta), ma su scala molto più contenuta di prima (era 167 vs 1138). I **Top 5 per distanza sono tutti outlier iniettati**: i 2 saldi estremi (1,34 M€ private, 986k mass) e tre dei ricchi-dormienti. Il metodo mette in cima esattamente i casi che abbiamo piantato.

Due avvertenze da tenere a mente:
- Il taglio χ² 0.975 ha un **tasso "di base" del ~2,5%** anche su dati perfettamente normali (è la sua frazione di falsi positivi per costruzione): parte di quel 9% è quello, non anomalia vera.
- Buona parte del resto sono i clienti **inattivi** (recency alta), una sotto-popolazione reale e bimodale, non errori. Se vuoi una *shortlist* stretta, alza la soglia (vedi grafico): a 0.999 restano 162 clienti, ordinandoli per distanza i primi sono i casi netti.

Il fondamentale grafico della MD interattivo:

In [ ]:
import plotly.graph_objects as go

d2 = d2_robusta                      # cambia in d2_classica per la versione "mascherata"
idx = np.arange(len(d2)); sopra = d2 > soglia
cols = ["id_cliente","segmento","saldo_medio_eur","transazioni_mese","giorni_ultima_operazione","abbandono"]
cd = df[cols].to_numpy()
ht = ("<b>%{customdata[0]}</b><br>segmento: %{customdata[1]}<br>"
      "saldo: %{customdata[2]:,.0f} €<br>txn/mese: %{customdata[3]}<br>"
      "gg ultima op.: %{customdata[4]}<br>abbandono: %{customdata[5]}<br>D²: %{y:.1f}<extra></extra>")

fig = go.Figure()
fig.add_trace(go.Scattergl(x=idx[~sopra], y=d2[~sopra], mode="markers", name="sotto soglia",
    marker=dict(size=4, color="#9fb3c8"), customdata=cd[~sopra], hovertemplate=ht))
fig.add_trace(go.Scattergl(x=idx[sopra], y=d2[sopra], mode="markers", name="outlier (> χ² 0.975)",
    marker=dict(size=5, color="crimson"), customdata=cd[sopra], hovertemplate=ht))

xmax = len(d2) - 1
for q, dash in [(0.90,"dot"),(0.95,"dash"),(0.975,"solid"),(0.99,"longdash")]:
    yv = chi2.ppf(q, p)
    fig.add_trace(go.Scatter(x=[0, xmax], y=[yv, yv], mode="lines+text",
        line=dict(color="#333", width=1.2, dash=dash),
        text=["", f"χ² {q:.3f} = {yv:.1f}"], textposition="top left",
        textfont=dict(size=10, color="#333"), name=f"χ² {q:.3f} = {yv:.1f}", hoverinfo="skip"))

fig.update_yaxes(type="log", title="Distanza di Mahalanobis al quadrato (D²)")
fig.update_xaxes(title="Cliente (indice nel dataset)")
fig.update_layout(title=f"Distanze di Mahalanobis robuste e soglie chi-quadro (p={p})",
                  height=540, hovermode="closest", legend=dict(orientation="h", y=1.02, x=0))
fig.show()

Conteggi sopra ciascuna soglia (robusta): **0.90 → 741, 0.95 → 500, 0.975 → 382, 0.99 → 291, 0.999 → 162**. Si vede a colpo d'occhio quanto la scelta del taglio sposti il numero di segnalati: passando il mouse sugli spike più alti trovi i clienti iniettati.

Come si legge. Sull'asse X ogni cliente al suo indice; in verticale la sua D² robusta in scala logaritmica — indispensabile, perché senza log gli eventuali spike potrebbero schiacciare tutto e le altre righe di soglia sarebbero invisibili. Le quattro linee orizzontali sono i quantili della chi-quadro a p=9 gradi di libertà: sono le soglie candidate.

I punti interattivi servono a questo: passando il mouse su uno spike leggiamo chi è.

**Quanti outlier "in totale"?**

Non esiste un numero unico: dipende dal metodo e dalla soglia. Sul dataset pulito:

| Metodo | Outlier | % |
| --- | --- | --- |
| IQR (unione su tutte le colonne) | ~567 | 13,5% |
| MAD (unione) | ~277 | 6,6% |
| Bivariata (combo) | 39 | 0,9% |
| Mahalanobis classica (0.975) | 116 | 2,8% |
| Mahalanobis robusta (0.975) | 382 | 9,1% |

L'unione su 11 colonne resta gonfiata dalla *molteplicità* (con tante variabili, molti clienti sono estremi su *qualcosa*): è un punto didattico, non un difetto del dato. Il nucleo solido e azionabile sono i ~40 del combo bivariato e i casi in cima alla distanza robusta — che corrispondono agli outlier piantati.

> 📘 **ground truth** — Nel file sono stati iniettati **18 outlier** di tre famiglie: *univariati estremi* (transazioni 95–130, saldo 0,9–1,5 M€, reclami 9–11, un'età 103 come errore di data entry); *multivariati "impossibili"* (giovani 22–25 anni con 24–28 anni di anzianità; private con 7 prodotti e patrimonio nullo); *combo ricco-dormiente* (saldo 120–180k con 0–1 transazioni e ~160 giorni di inattività). Recall: IQR li riprende 15/18, il combo bivariato 5/18, la Mahalanobis robusta 16/18 — e ogni iniettato è ripreso da almeno un metodo. Nessun metodo da solo li prende tutti: è la lezione operativa (univariato e multivariato guardano cose diverse, vanno usati insieme).

# Componente A — Classificazione

Il flusso di lavoro tipico, dal dato al modello interpretabile:

1. Preparazione dati: gestione dei missing (es. nps), codifica delle categoriali (segmento, canale) con one-hot encoding, esclusione di id_cliente e della colonna testuale.
2. Split train/test stratificato, per mantenere la stessa proporzione di abbandoni nei due insiemi.
3. Addestramento di uno o più modelli: **Logistic Regression** (semplice e interpretabile, ottima baseline), **Random Forest / Gradient Boosting** (più potenti).
4. Valutazione con metriche adatte a dati sbilanciati.
5. Interpretazione: quali "feature" (aka, "predittori", ecc) pesano di più (feature importance / SHAP). È l’input della Componente B.


In [ ]:
# cella di PRE-PROCESSING dei dati di input al modello:

X = df.drop(columns=["id_cliente", "abbandono", "note_gestore"]) # il parametro 'inplace' è qui inutile pewrchè creaiamo un nuovo df 'X'
# la colonna 'Id' è inutile, la colonnna 'abbandono' non un preditore, la colonna testuale 'note_gestore' viene gestita a parte
# (vedi Componente B)
# la funzione 'get_dummies' di pandas fa hot encoding (un passo di pre-elaborazione delle variabili categoriche)
X = pd.get_dummies(X, columns=["segmento", "canale_preferito"], drop_first=True) # X maiuscolo perchè è un df
y = df["abbandono"]   # la colonna risposta - y minuscolo perchè è una serie
print(X.head())
y.head()

# Split del dataset

In [ ]:
# 0.25 come % di test è un valore tipico
# random_state: per riproduzione su ogni PC / esecuzione
# stratify = Yes crea subset (train e test) con la stessa proporzioni di abbandoni
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=7
)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
y_test

# Allenamento del modello (fit)

In [ ]:
# chaining configurazione + fit (circa 3-4")
# il classificatore 'RandomForestClassifier' è uno dei migliori, ha una serie di parametri
modello = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=7
).fit(X_train, y_train)   # fatto solo sul training

In [ ]:
modello

In [ ]:
type(modello)

# Esame dei risultati

In [ ]:
prob = modello.predict_proba(X_test)  # P(mantenimento) e P(abbandono) per ogni cliente
print(type(prob))
prob # un'array numpy

La matrice di confusione (la suddivisione nei 4 esiti possibili: TP, TN, FP, FN):

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

# Previsione con cutoff = 0.5
# Previsione con cutoff = 0.5 sulla probabilità di abbandono

#  il vettore delle previsioni
# y_pred = (prob[:, 1] >= 0.5).astype(int) # setta 1 (abbandono)

y_pred = np.where(
    prob[:, 1] >= 0.3,
    "abbandono",
    "non_abbandono"
)

print(y_pred)

y_test_str = np.where(
    y_test == 1,
    "abbandono",
    "non_abbandono"
)

# Matrice di confusione
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test_str,
    y_pred,
    display_labels=["abbandono", "non_abbandono"]
)




In [ ]:
auc = roc_auc_score(y_test, prob[:, 1])
fpr, tpr, _ = roc_curve(y_test, prob[:, 1])

plt.plot(fpr, tpr, label=f"Random Forest (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "--", color="grey", label="Caso casuale (AUC = 0.5)")
plt.xlabel("Tasso di falsi positivi")
plt.ylabel("Tasso di veri positivi (recall)")
plt.title("Curva ROC - modello di abbandono")
plt.legend()
plt.show()

Figura 2 — Curva ROC prodotta dal codice qui sopra: il Random Forest (AUC 0,774) si stacca nettamente dalla diagonale del caso casuale. Come leggerla, nella sezione « Le metriche, spiegate ».

In [ ]:
Image('ROC-AUC1.png') if IN_COLAB else display(Image(filename='ROC-AUC1.png'))

In [ ]:
Image('ROC-AUC2.png') if IN_COLAB else display(Image(filename='ROC-AUC2.png'))

### Il punto chiave: dati sbilanciati e scelta della soglia

Su questo dataset un Logistic Regression raggiunge un ROC-AUC di circa 0,80 e un Random Forest di circa 0,77 (un classificatore casuale varrebbe 0,50): c’è segnale reale da imparare. Ma la metrica decisiva è un’altra.

> **L’esperimento che apre gli occhi**
>
> Con la soglia di default **0,50**, il Random Forest segnala pochissimi clienti e intercetta solo **6 abbandoni su 148** (recall ≈ 4%): un modello che « sembra » funzionare, ma in pratica è inutile per il business.
> Abbassando la soglia a **0,30**, ne intercetta **43 su 148** (recall ≈ 29%), al costo di 65 falsi allarmi. La soglia è una leva di business.

Da qui il concetto centrale per gli analisti: scegliere la soglia è una decisione di business, non statistica. Bisogna pesare il costo dei due errori:

- **Falso negativo** (non segnalo un cliente che poi abbandona): cliente perso, AUM e ricavi persi → **costo alto**.
- **Falso positivo** (segnalo un cliente fedele): tempo del gestore sprecato → **costo basso/medio**.

Poiché perdere un cliente costa molto più di una telefonata « a vuoto », conviene una soglia bassa, che privilegia il recall. Le metriche da guardare sono quindi recall, precision, F1 e PR-AUC, più che la sola accuracy — ingannevole su dati sbilanciati: un modello che dicesse « nessuno abbandona » avrebbe già l’86% di accuratezza, pur essendo inservibile.

### Le metriche, spiegate

Le metriche appena citate meritano una spiegazione per un pubblico non tecnico, ancorata al caso.

**Curva ROC e AUC**. La curva ROC mostra, al variare della soglia, quanti abbandoni reali intercettiamo (asse verticale, il recall) a fronte di quanti falsi allarmi generiamo (asse orizzontale). L’AUC è l’area sotto quella curva, riassunta in un numero: la si può leggere come la probabilità che, presi a caso un cliente che abbandona e uno che resta, il modello assegni un rischio più alto al primo. 0,5 = caso (inutile), 1,0 = separazione perfetta; qui il Random Forest è a 0,77.

**PR-AUC**. Su dati sbilanciati l’AUC può essere lusinghiera, perché premia anche l’ordinare bene i tanti clienti che restano. La PR-AUC guarda invece solo alla classe rara — gli abbandoni — e riassume in un numero il compromesso tra precision e recall su tutte le soglie. Differenza cruciale: il suo valore di riferimento non è 0,5 ma la **prevalenza** della classe positiva, qui 0,141. Il nostro modello è a 0,35, circa 2,5× il caso casuale: va sempre letta contro la prevalenza, mai contro 0,5.

**Recall e precision**. Si leggono sulla stessa matrice di confusione. Alla soglia 0,30 (43 abbandoni intercettati, 65 falsi allarmi): il _recall_ è la copertura — tra chi abbandona davvero, quanti ne intercetto (43 su 148 = 29%, risponde a « quanti me ne sfuggono »); la _precision_ è la precisione — tra chi segnalo, quanti abbandonano davvero (43 su 108 = 40%, risponde a « quando suono l’allarme, quanto spesso ho ragione »). Le due sono in tensione, governata dalla soglia: abbassarla alza il recall ma fa scendere la precision. Quale privilegiare dipende dai costi dei due errori — la decisione di business vista sopra.

**F1**. La media armonica di precision e recall: un voto unico quando si vogliono bilanciare copertura e precisione in un solo numero.

> È la definizione dell'**F1**. La media armonica di due numeri *a* e *b* è `2ab / (a + b)`, quindi:

```
  F1 = 2 · (precision · recall) / (precision + recall)
```

Il motivo per cui si usa la media **armonica** e non la solita media aritmetica è il punto chiave: l'armonica **penalizza lo squilibrio** tra i due valori. È alta solo se *entrambe* precision e recall sono alte; se una delle due crolla, trascina giù tutto l'F1 verso il valore basso.

Esempio che lo rende evidente — un modello che segnala pochissimi clienti ma quasi sempre giusti: precision 0,95, recall 0,05.
- Media **aritmetica**: (0,95 + 0,05)/2 = **0,50** → sembra discreto, ma è ingannevole (sta perdendo il 95% degli abbandoni).
- Media **armonica (F1)**: 2·0,95·0,05 / (1,00) = **0,095** → correttamente pessimo.

Sul nostro caso churn a soglia 0,30 (precision ≈ 0,40, recall ≈ 0,29) viene F1 = 2·0,40·0,29/0,69 ≈ **0,34**: qui i due valori sono vicini, quindi armonica e aritmetica quasi coincidono — è quando divergono che l'armonica fa il suo lavoro. In una riga: l'F1 è un **voto unico** che ti costringe a essere bravo su precision *e* recall insieme, senza poter compensare l'una con l'altra.

In codice, una volta scelta la soglia, le tre metriche si ottengono con le funzioni dedicate di scikit-learn (riusando `prob` e `y_test`):


In [ ]:
Image('ROC-AUC3.png') if IN_COLAB else display(Image(filename='ROC-AUC3.png'))

In [ ]:
# Trasformiamo le probabilità in decisioni 0/1 applicando la soglia
soglia = 0.30
predizioni = (prob[:, 1] >= soglia).astype(int)

print(f"Recall:    {recall_score(y_test, predizioni):.2f}")    # ~0.29
print(f"Precision: {precision_score(y_test, predizioni):.2f}") # ~0.40
print(f"F1:        {f1_score(y_test, predizioni):.2f}")        # ~0.34

`classification_report` è una funzione di *scikit-learn* (sta dentro *sklearn.metrics*) che, data la verità (`y_test`) e le predizioni del modello, stampa in un colpo solo **tutte le metriche di classificazione per ogni classe**: precision, recall, F1 e support.

Cambiando soglia i numeri cambiano: è il legame diretto con la decisione vista sopra. Per avere lo stesso quadro su entrambe le classi in un colpo solo c’è classification_report:


In [ ]:
# Stesso quadro, per entrambe le classi, in un colpo solo
print(classification_report(y_test, predizioni,
                            target_names=["resta", "abbandona"]))

# Esame dell'importanza delle feature

### Interpretabilità: dalla feature importance a SHAP

Il modello non solo predice: dice anche quali feature pesano. Ma ci sono due livelli, da non confondere — ed è il secondo che alimenta la spiegazione dell’LLM.

**Feature importance — globale**. È senza segno e vale su tutti i clienti: dice quali variabili contano per l’abbandono in generale. Non dice perché un singolo cliente è a rischio.


In [ ]:
# quali feature pesano di più per il modello, in generale (vista globale)
importanze = pd.Series(modello.feature_importances_, index=X.columns)
importanze = importanze.sort_values(ascending=False)
print(importanze.head(10))   # in testa: transazioni_mese, anzianita_mesi, nps, ...

Usiamo ora `TreeExplainer`, la classe di SHAP specializzata nel calcolare [i valori SHAP](https://en.wikipedia.org/wiki/Shapley_value) per i modelli ad albero: Random Forest (il primo caso), Gradient Boosting, XGBoost, LightGBM, alberi singoli.

**SHAP — locale**. Per un singolo cliente, scompone la sua probabilità di abbandono nei contributi delle singole feature: quanto ciascuna ha spinto la previsione verso l’alto (+) o verso il basso (−), rispetto al cliente medio. I contributi hanno un segno, sono personalizzati sul cliente (tengono conto di tutte le sue feature insieme, non di una alla volta) e sommati ricostruiscono la previsione. È la materia prima onesta per l’LLM: non « le feature importanti in generale », ma « perché questo cliente ». Qui sotto il codice per calcolare e mostrare i contributi di un cliente; nel Ruolo 1 della Componente B questi stessi contributi diventano l’input dell’LLM.


In [ ]:
# circa 100"-120" per la costruzione dell'explainer (la cella più lenta del notebook)

explainer = shap.TreeExplainer(modello)        # 1) "prepara" l'Explainer (lo spiegatore) del tuo modello
shap_values = explainer.shap_values(X_test)    # 2) calcola i contributi per quei clienti (di test)

Due righe, due fasi: prima costruiamo l'oggetto `explainer` legandolo al modello addestrato, poi gli diamo dei dati (`X_test`)
> e lui restituisce, per ogni cliente e ogni feature, quanto quella feature ha spinto la previsione verso l'alto o verso il basso.

---
**📌  Nota sugli *Explainer* di SHAP**

100-120 secondi per "spiegare" un modello sembrano tanti. Il motivo sta in **quanto lavoro** TreeSHAP deve fare, che è il prodotto di più fattori tutti presenti in questo caso.

Il costo di `explainer.shap_values(X_test)` cresce, in modo semplificato, come

> **(numero di clienti da spiegare) × (numero di alberi) × (dimensione di ogni albero)**

Nel nostro caso:
- **1.050 clienti** nel test set — e SHAP calcola i contributi *per ognuno separatamente*, perché la spiegazione è locale, su misura del singolo cliente. Non è un conto fatto una volta sola: **è fatto 1.050 volte**.
- **300 alberi** (`n_estimators=300`): per ogni cliente, SHAP deve attraversare *tutti e 300* gli alberi della foresta e combinare i loro contributi.
- **alberi grandi**: con `class_weight="balanced"` e nessun limite di profondità, ogni albero cresce parecchio (molti nodi). E il costo di TreeSHAP per albero non dipende solo dal numero di foglie, ma cresce con la **profondità al quadrato** (≈ foglie × profondità²) — è la parte che fa lievitare il tutto.

Moltiplica: 1.050 × 300 × (alberi profondi) = parecchie centinaia di milioni di operazioni. Da qui i ~120 secondi.

Va detta una cosa per dare la giusta prospettiva: **TreeSHAP, l'algoritmo usato da `TreeExplainer`, è già l'algoritmo veloce**. Quei 120 secondi sono il prezzo della spiegazione *esatta*. Con `KernelExplainer` (quello generico a scatola nera) sugli stessi dati non parleremmo di minuti ma di **ore o peggio** — quello è il confronto da tenere a mente.

Se in futuro servisse più rapidità, le leve sono proprio quei fattori:
- **spiegare meno clienti.** Quasi sempre non servono tutti e 1.050: per il caso d'uso (mostrare al gestore *perché* un cliente è a rischio) ne basta uno, o una manciata. `explainer.shap_values(X_test.iloc[[0]])` su un solo cliente è quasi istantaneo. È la leva di gran lunga più efficace.
- **meno alberi o alberi meno profondi** (es. `max_depth`), ma questo tocca il *modello* e ne può ridurre l'accuratezza — non lo si farebbe solo per velocizzare SHAP.

Nel notebook stiamo spiegando l'intero `X_test` (1.050 clienti) per avere il quadro completo durante la didattica, ed è giusto così — ma è anche il motivo dei 120 secondi. In un uso "reale" su un singolo cliente alla volta, SHAP risponde in un lampo.


---
Perché `TreeExplainer`, specializzato negli alberi, e non l'explainer generico di shap (`KernelExplainer`), che va bene per qualsiasi modello?

`KernelExplainer`funziona con qualunque modello trattandolo come una scatola nera (gli dà degli input, guarda gli output, e da lì stima i contributi). E' lento, perché per stimare i valori di Shapley deve interrogare il modello moltissime volte provando tante combinazioni di feature.

`TreeExplainer` invece è specializzato: funziona solo sui modelli ad albero (come il Random Forest); proprio perché conosce la struttura interna dell'albero riesce a calcolare gli stessi valori in modo esatto e molto più veloce. Niente scatola nera: guarda direttamente come è fatto l'albero.
E' infinitamente più rapido (i ~100 secondi invece di tempi enormi).

SHAP nasce dalla teoria dei giochi (i valori di Shapley), e calcolarli esattamente in generale è proibitivo: richiederebbe di provare tutte le combinazioni di feature, un costo che esplode. Per i modelli ad albero, però, esiste un algoritmo (TreeSHAP) che li calcola in modo esatto ed efficiente, sfruttando la struttura dell'albero (i percorsi, gli split, quanti campioni passano da ogni nodo). TreeExplainer è l'implementazione di quell'algoritmo. È il motivo per cui sul tuo Random Forest gira in un tempo ragionevole (i ~100 secondi che avevi annotato) invece di metterci ore.

SHAP ha altri "explainer" per altri tipi di modello — per esempio `LinearExplainer` per i modelli lineari, `KernelExplainer` (generico ma lento, funziona con qualsiasi modello trattandolo come scatola nera), `DeepExplainer/GradientExplainer` per le reti neurali. Si usa `TreeExplainer` semplicemente perché il modello è una foresta di alberi: è la scelta giusta e la più veloce per quel caso.

NB. Per un classificatore binario, `TreeExplainer` restituisce i contributi per entrambe le classi — ed è proprio per questo che nel codice c'è la riga che estrae la classe "abbandono" (l'indice 1), gestendo le due forme diverse a seconda della versione di shap.

---

In [ ]:
# Contributi della classe "abbandono" (= 1). A seconda della versione di shap,
# shap_values può essere una lista [classe_0, classe_1] (API classica) oppure
# un array (campioni, feature, classi) (API recente): gestiamo entrambi i casi.
if isinstance(shap_values, list):
    sv_churn = shap_values[1]
else:
    sv = np.asarray(shap_values)
    sv_churn = sv[:, :, 1] if sv.ndim == 3 else sv

In [ ]:
# Contributi per un singolo cliente: quanto ogni feature ha spinto il suo rischio
pos = 0   # il primo cliente del test, come esempio
contributi = pd.Series(sv_churn[pos], index=X_test.columns).sort_values(key=abs, ascending=False)
print(contributi.head(8))   # +  alza il rischio,   -  lo abbassa

> **Compatibilità tra versioni di shap**
>
> L’output di shap_values per la classificazione binaria dipende dalla versione della libreria: una lista [classe_0, classe_1] in quelle classiche, un array (campioni, feature, classi) in quelle recenti. Lo snippet gestisce entrambi i casi.


Come leggerlo: `pos = 0` prende **il primo cliente del test set**, che è un cliente <u>a basso rischio</u> — e infatti **tutti e otto i contributi principali sono negativi**, cioè **ogni feature lo sta spingendo lontano dall'abbandono**.<br>
> La convenzione di segno è quella: valore negativo abbassa il rischio, positivo lo alza.

Per questo cliente il fattore più protettivo è `accessi_app_mese` (-0,095): il cliente usa molto l'app, e l'engagement digitale è il segnale che più lo trattiene; seguono NPS alto, stipendio accreditato, transazioni — tutte caratteristiche "buone".

Per questo cliente il modello non vede fattori di rischio, vede fattori di fidelizzazione.

Per vedere il **caso opposto e più istruttivo** — un cliente dove **i contributi sono positivi perché lo spingono verso l'abbandono** — ci basta puntare `pos` sul **cliente più a rischio** invece che sul primo. Aggiungiamo una riga prima del calcolo:

In [ ]:
rischio = pd.Series(prob, index=X_test.index)
pos = X_test.index.get_loc(rischio.idxmax())   # il cliente più a rischio invece del primo

e rilanciamo la cella di calcolo di prima: lì vedremo in testa, con segno positivo, `transazioni_mese basse`, tanti `giorni_ultima_operazione`, `stipendio_accreditato` mancante — gli stessi driver che nelle celle successive vengono passati all'LLM.

È il modo migliore per vedere "dal vivo" il legame tra SHAP e la spiegazione che il modello manda al gestore.


In [ ]:
contributi = pd.Series(sv_churn[pos], index=X_test.columns).sort_values(key=abs, ascending=False)
print(contributi.head(8))   # +  alza il rischio,   -  lo abbassa

# Arricchimento con LLM

# Componente B — LLM via API

Una volta che il modello ha prodotto, per ogni cliente a rischio, la probabilità e i principali driver, l’LLM interviene in tre ruoli.

Tutte le chiamate seguenti usano un client già configurato. La chiave API si acquisisce una volta sola: in Google Colab si legge dai Secret del notebook, così non finisce mai scritta nel codice.


In [ ]:
# In Colab la chiave si salva nei Secret (icona a forma di chiave nel menu a sinistra),
# con nome OPENAI_API_KEY e accesso abilitato per questo notebook.
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

### Ruolo 1 — Spiegazione in linguaggio naturale

L’LLM riceve i fattori di rischio (estratti dal modello) e li trasforma in 3-4 frasi che il gestore legge in pochi secondi. Si passano solo i fattori reali e si istruisce esplicitamente il modello a non inventare dati: è il primo presidio contro le allucinazioni.


In [ ]:
def spiega_rischio(cliente: dict, prob: float, driver: list[dict]) -> str:
    """Trasforma l'output del modello in una spiegazione per il gestore."""
    fattori = "\n".join(
        f"- {d['feature']}: {d['valore']} (impatto {d['impatto']})"
        for d in driver
    )
    prompt = (
        "Sei un assistente per i gestori di una banca.\n"
        f"Cliente {cliente['id_cliente']} - probabilità di abbandono: {prob:.0%}.\n"
        f"Principali fattori che spingono il rischio:\n{fattori}\n\n"
        "Scrivi 3-4 frasi in italiano che spieghino al gestore perché "
        "il cliente è a rischio. Tono professionale, niente gergo tecnico. "
        "Non inventare dati che non sono elencati sopra."
    )
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    return resp.choices[0].message.content

Definita la funzione, ecco come la si alimenta: è il punto in cui l’output del modello diventa l’input dell’LLM (il cablaggio cervello→voce). Si sceglie un cliente a rischio e si costruiscono i tre input — cliente, probabilità e driver.

> **Attenzione: i driver devono essere fattori che davvero alzano il rischio**
>
> Prendere le feature più importanti del modello ed etichettarle « sopra/sotto la media » è un errore. Per un cliente, anzianità alta e NPS alto sono sopra la media ma **riducono** il rischio, non lo aumentano: passarli all’LLM come « fattori di rischio » lo porterebbe a inventare una motivazione — proprio l’allucinazione che vogliamo evitare. Si selezionano quindi solo le feature in cui il cliente è sul lato critico, usando la direzione del rischio appresa dai dati.


In [ ]:
# 1) Scegliamo un cliente ad alto rischio tra quelli del test
rischio = pd.Series(prob[:, 1], index=X_test.index)
idx = rischio.idxmax()                       # il cliente con probabilità più alta
prob_cliente = rischio.loc[idx]

# 2) I dati del cliente che servono nel prompt
cliente = {
    "id_cliente": df.loc[idx, "id_cliente"],
    "segmento":   df.loc[idx, "segmento"],
}

# 3) I driver: le feature più importanti per il modello, ma SOLO quelle in cui
#    questo cliente è davvero sul lato "a rischio". La direzione del rischio la
#    apprendiamo dai dati (segno della correlazione col target); per un'attribuzione
#    rigorosa per singolo cliente si usano i valori SHAP (vedi sotto).
importanze = pd.Series(modello.feature_importances_, index=X.columns)
mediane    = df.median(numeric_only=True)
direzione  = df.corr(numeric_only=True)["abbandono"]   # > 0: valore alto = più rischio

driver = []
for f in importanze.sort_values(ascending=False).index:
    if f not in df.columns or f not in direzione.index:
        continue                              # salta le colonne create da get_dummies
    valore = df.loc[idx, f]
    alto_e_rischioso = direzione[f] > 0
    sul_lato_rischioso = valore > mediane[f] if alto_e_rischioso else valore < mediane[f]
    if sul_lato_rischioso:
        verso = "valore alto" if alto_e_rischioso else "valore basso"
        driver.append({"feature": f, "valore": round(float(valore), 1), "impatto": verso})
    if len(driver) == 3:
        break

# 4) Chiamiamo l'LLM
spiegazione = spiega_rischio(cliente, prob_cliente, driver)
print(spiegazione)

Versione rigorosa: invece della direzione marginale (correlazione), i valori SHAP danno il contributo di ogni feature condizionato a tutto il profilo del cliente. Richiede la libreria shap.


In [ ]:
# Driver per l'LLM: i 3 fattori che più AUMENTANO il rischio di questo cliente

pos = X_test.index.get_loc(idx)                                    # ricava nuovamente 'pos' da 'idx', così è sempre il cliente giusto a prescindere da cosa si è eseguito prima;
contributi = pd.Series(sv_churn[pos],
                      index=X_test.columns).sort_values(key=abs,
                      ascending=False)                             # ricava pos da idx, così è sempre il cliente giusto a prescindere da cosa hai eseguito prima;

driver = []
for f in contributi.index:
    if contributi[f] <= 0:        # teniamo solo i contributi che alzano il rischio
        continue
    driver.append({
        "feature": f,
        "valore":  df.loc[idx, f] if f in df.columns else "(categoria)",
        "impatto": f"+{contributi[f]:.3f} sul rischio",
    })
    if len(driver) == 3:
        break

spiegazione = spiega_rischio(cliente, prob_cliente, driver)
print(spiegazione)




> **Compatibilità tra versioni di shap**
>
> Come nel Cap. 5, la riga che estrae la classe « abbandono » gestisce entrambi i formati di shap_values (lista per-classe nelle versioni classiche, array a tre dimensioni in quelle recenti), quindi funziona a prescindere dalla versione.


Un LLM restituisce il testo come un unico paragrafo senza "a capo", e lo *stdout* di Colab non va a capo da solo: quindi produce una riga lunghissima con lo scroll orizzontale.

[textwrap](https://docs.python.org/3/library/textwrap.html) è un modulo della libreria standard di Python (non dobbiamo installarlo, c'è già) che serve a **formattare blocchi di testo**: la cosa che si usa più spesso è mandare a capo una stringa lunga a una larghezza fissa, in modo che non sfori.

In [ ]:
# semplice stampa alternativa
print(textwrap.fill(spiegazione, width=90))

Due cose da sapere su questa cella, diverse dalla precedente cella SHAP:

* dipende da `idx`, `cliente` e `prob_cliente`, che vengono definiti nella cella subito prima (quella che costruisce il driver per direzione di rischio). Assicurarsi di averla eseguita, altrimenti `idx` non esiste.
* l'ultima riga, `spiega_rischio(...)`, fa una chiamata reale all'LLM — quindi qui serve la chiave nei Secret.

---
**Nota su `textwrap`**:

`textwrap.fill(testo, width=90)` prende la stringa spiegazione — che l'LLM restituisce come un unico paragrafo senza "a capo" — e la riscrive inserendo gli "a capo" in modo che nessuna riga superi i 90 caratteri, spezzando solo tra una parola e l'altra (non a metà parola).

Il risultato è il testo incolonnato e leggibile che abbiamo ottenuto, invece dell'unica riga lunghissima con la barra di scorrimento orizzontale.

Il `width=90` è semplicemente il numero massimo di caratteri per riga: si può abbassare (es. 70) per colonne più strette, alzare per righe più lunghe.
Due cose della stessa famiglia, se servono:
* `textwrap.fill(testo, width=90)` → restituisce una stringa già con gli a capo (quella che usiamo).
* `textwrap.wrap(testo, width=90)` → fa la stessa divisione ma restituisce una lista di righe (utile se vogliamo lavorarci sopra invece di stamparle).

È puramente cosmetico: non tocca il contenuto della spiegazione, cambia solo come viene mandata a video. Serve perché `print()` da solo non manda a capo, e Colab quindi ci mostrerebbe tutto su una riga sola.

---

### Ruolo 2 — Generazione dell’azione di retention

L’LLM propone un’azione concreta (canale, messaggio, priorità) in formato JSON strutturato, così da essere direttamente integrabile in un sistema (CRM, cruscotto del gestore) e non solo letto in chat.


In [ ]:
def proponi_azione(cliente: dict, prob: float, motivo: str) -> dict:
    """Genera un'azione di retention come oggetto strutturato (JSON)."""
    prompt = (
        f"Cliente {cliente['id_cliente']}, segmento {cliente['segmento']}, "
        f"rischio di abbandono {prob:.0%}. Motivo principale: {motivo}.\n\n"
        "Proponi UNA azione di retention concreta e specifica. "
        "Rispondi SOLO con JSON valido nel formato:\n"
        '{"canale": "...", "messaggio": "...", "priorita": "alta|media|bassa"}'
    )
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},   # output garantito in JSON
        temperature=0.4,
    )
    return json.loads(resp.choices[0].message.content)  # json.loads ("load string") prende la stringa JSON — es. {"canale": "...", "priorita": "alta"} —
                                                        # e la restituisce come dizionario, così si può accedere ai campi con azione["canale"], azione["priorita"], ecc.

La chiamata riusa il cliente, la probabilità e i driver già calcolati per il Ruolo 1:


In [ ]:
# riusiamo cliente, prob_cliente e i driver già calcolati per il Ruolo 1.
# il "motivo" è il fattore di rischio principale, col suo valore: passare il valore
# (es. "transazioni_mese = 3") evita che l'LLM lo interpreti male e inventi.
motivo = f"{driver[0]['feature']} = {driver[0]['valore']}" if driver else "attività in calo"


azione = proponi_azione(cliente, prob_cliente, motivo)

print(f"Azione per il cliente {cliente['id_cliente']}:")
print(f"  canale:   {azione['canale']}")
print(f"  priorità: {azione['priorita']}")
print(f"  messaggio: {azione['messaggio']}")

Nota: proponi_azione restituisce un dizionario (l’LLM è forzato a JSON con response_format), quindi si accede ai campi con azione["canale"] invece di leggere testo libero — è ciò che rende l’output integrabile in un sistema.


### Ruolo 3 (estensione) — L’LLM come estrattore di feature dal testo

Le note libere del gestore contengono informazioni preziose ma non strutturate (« ha menzionato un’offerta di un concorrente »). L’LLM le trasforma in feature strutturate (sentiment, menzione di un concorrente, tema) da reimmettere nel modello. È il pattern « testo non strutturato → dati strutturati », frequentissimo nel lavoro reale.


In [ ]:
def estrai_segnali(nota: str) -> dict:
    """Estensione: l'LLM trasforma testo libero in feature strutturate."""
    if not isinstance(nota, str) or not nota.strip():
        return {"sentiment": "n/d", "menziona_concorrente": 0, "tema": "n/d"}
    prompt = (
        "Analizza la nota del gestore e rispondi SOLO con JSON:\n"
        '{"sentiment": "positivo|neutro|negativo", '
        '"menziona_concorrente": 0 o 1, '
        '"tema": "costi|servizio|investimenti|nessuno|altro"}\n\n'
        f'Nota: "{nota}"'
    )
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0,
    )
    return json.loads(resp.choices[0].message.content)

La chiamata su un singolo cliente:


In [ ]:
# Prendiamo un cliente che ha davvero una nota del gestore.
con_nota = df[df["note_gestore"].notna() & (df["note_gestore"].str.strip() != "")].iloc[0]

nota = con_nota["note_gestore"]
segnali = estrai_segnali(nota)

print(f"Nota:     {nota}")
print(f"Estratto: {segnali}")
# -> es. {'sentiment': 'negativo', 'menziona_concorrente': 1, 'tema': 'investimenti'}

L’esempio sopra estrae i segnali da un solo cliente. Il passo che rende l’estensione davvero utile è applicarla all’intera colonna, trasformando il testo in nuove feature del modello:


In [ ]:
# circa 20-25'
# (OPZIONALE) Da nota libera a colonne strutturate, su tutti i clienti.
# Trasformiamo il testo del gestore in nuove feature utilizzabili dal modello.
estratti = df["note_gestore"].apply(estrai_segnali).apply(pd.Series)
estratti = estratti.add_prefix("nota_")     # nota_sentiment, nota_menziona_concorrente, nota_tema

df_esteso = pd.concat([df, estratti], axis=1)
df_esteso[["id_cliente", "note_gestore",
           "nota_sentiment", "nota_menziona_concorrente", "nota_tema"]].head()

In [ ]:
# versione ridotta
campione = df.head(20).copy()                 # solo le prime 20 righe
estratti = campione["note_gestore"].apply(estrai_segnali).apply(pd.Series).add_prefix("nota_")
df_esteso = pd.concat([campione, estratti], axis=1)
df_esteso[["id_cliente", "note_gestore",
           "nota_sentiment", "nota_menziona_concorrente", "nota_tema"]].head()

Le tre colonne nota_sentiment, nota_menziona_concorrente e nota_tema si aggiungono alle feature numeriche e rientrano nello stesso get_dummies e addestramento della Componente A: così un segnale che prima viveva solo nel testo libero (« ha accennato a trasferire i dossier titoli ») diventa un dato che il modello può pesare. Il cerchio si chiude, dall’LLM al modello.

> **Costo e latenza**
>
> Una chiamata LLM per ogni riga ha un costo (tempo e denaro) e una latenza non trascurabili. Su 4.200 clienti è una dimostrazione; su milioni di clienti l’estrazione si fa in batch, una volta sola, salvando il risultato — non si rilancia a ogni addestramento.

### Tre principi trasversali

- **Output strutturato**: per usare l’LLM dentro un processo, chiedere JSON e validarlo, non testo libero.
- **Controllo delle allucinazioni**: passare solo dati reali nel prompt, istruire il modello a non inventarne; temperatura bassa per i compiti deterministici (estrazione).
- **Human-in-the-loop**: spiegazione e azione sono suggerimenti; la decisione e il contatto con il cliente restano sempre al gestore. Fondamentale in un contesto bancario regolamentato.


# Allucinazioni

Un LLM tende a **riempire i vuoti**: se gli si chiede qualcosa di specifico ma non gli si danno i dati, produce comunque un testo fluente e sicuro di sé — **inventando** i dettagli che mancano. È il rischio centrale quando lo si usa per il business, e finora lo abbiamo tenuto a bada con tre accorgimenti dentro la funzione `spiega_rischio`: passargli i **dati reali con i valori**, l'istruzione **"non inventare"**, e una **temperatura bassa**.

Qui facciamo il contrario, **di proposito**: togliamo tutte e tre le difese e gli chiediamo episodi e reclami specifici che nei nostri dati non esistono. Guardiamo cosa succede.

### Allucinazioni: vederle da vicino

Un LLM tende a riempire i vuoti: se gli si chiede qualcosa di specifico ma non gli si forniscono i dati, produce comunque un testo fluente e sicuro di sé, inventando i dettagli mancanti. È il rischio centrale nell’uso per il business, e finora lo abbiamo tenuto a bada con tre accorgimenti dentro spiega_rischio: passare i dati reali con i valori, l’istruzione « non inventare » e una temperatura bassa. Qui facciamo il contrario, di proposito: togliamo tutte e tre le difese e chiediamo episodi e reclami specifici che nei dati non esistono.


In [ ]:
# Versione SENZA difese, solo per mostrare l'allucinazione:
# - non riceve i dati reali del cliente (solo id e probabilità)
# - non ha il freno "non inventare"
# - temperatura alta (più "fantasia")
# - e per di più le chiediamo dettagli specifici che NON esistono nei dati
def spiega_rischio_ingenua(cliente: dict, prob: float) -> str:
    prompt = (
        "Sei un assistente per i gestori di una banca.\n"
        f"Il cliente {cliente['id_cliente']} ha una probabilità di abbandono del {prob:.0%}.\n\n"
        "Scrivi un paragrafo per il gestore che spieghi perché questo cliente è insoddisfatto, "
        "descrivendo episodi concreti, reclami specifici e contatti recenti con la banca."
    )
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0,
    )
    return resp.choices[0].message.content

# Stesso cliente a rischio di prima (cliente e prob_cliente sono già definiti sopra)
print(textwrap.fill(spiega_rischio_ingenua(cliente, prob_cliente), width=90))

Confrontando il paragrafo prodotto con ciò che sappiamo davvero del cliente, il problema salta agli occhi: i nostri dati contengono solo numeri — transazioni, giorni dall’ultima operazione, NPS, saldo — e nessuna traccia di reclami, telefonate o episodi in filiale. Eppure l’LLM li ha descritti con sicurezza: li ha inventati. È successo perché abbiamo tolto le tre difese:

- **Niente dati reali nel prompt: **solo l’ID e la probabilità. Non avendo nulla di concreto da dire, il modello crea qualcosa di concreto.
- **Niente « non inventare »: **senza quel freno esplicito, riempire i vuoti è il comportamento naturale.
- **Temperatura alta (1.0): **più libertà, più fantasia. Per i compiti fattuali si tiene bassa (qui usavamo 0,3).

E in più gli abbiamo chiesto episodi e reclami specifici: una richiesta che, senza dati, può essere soddisfatta solo inventando. La spiega_rischio « buona », sullo stesso cliente, diceva invece soltanto « 3 transazioni, 58 giorni dall’ultima operazione, niente stipendio accreditato » — tutto vero — proprio perché aveva le tre difese. È lo stesso meccanismo, in forma più subdola, già visto nel Ruolo 2: lì un motivo criptico faceva inventare all’LLM un « aumento delle transazioni ».

> **La regola pratica**
>
> L’output di un LLM è sempre un suggerimento da verificare, mai una verità. Le difese riducono il rischio, ma il controllo finale resta all’umano — il human-in-the-loop. In un contesto bancario regolamentato, è la differenza tra uno strumento utile e un guaio.


**Cosa è appena successo**

Rileggiamo il paragrafo qui sopra e confrontiamolo con quello che **sappiamo davvero** di questo cliente. I nostri dati contengono solo numeri: transazioni, giorni dall'ultima operazione, NPS, saldo… **Non c'è alcuna traccia** di reclami, telefonate, episodi in filiale o lamentele specifiche. Eppure l'LLM li ha **descritti con sicurezza**: li ha inventati di sana pianta.

**Perché è successo** — abbiamo tolto le tre difese:

1. **Niente dati reali nel prompt.** Gli abbiamo dato solo l'ID e la probabilità. Non avendo nulla di concreto da dire, il modello *crea* qualcosa di concreto.
2. **Niente "non inventare".** Senza quel freno esplicito, riempire i vuoti è il comportamento naturale.
3. **Temperatura alta (1.0).** Più libertà = più fantasia. Per i compiti fattuali si tiene bassa (noi usavamo 0.3).

E in più gli abbiamo **chiesto** episodi e reclami specifici: una richiesta che, senza dati, può essere soddisfatta solo inventando.

**Il contrasto.** La funzione `spiega_rischio` "buona" (celle sopra) sullo stesso cliente diceva solo *"3 transazioni, 58 giorni dall'ultima operazione, niente stipendio accreditato"* — tutto vero — proprio perché aveva le tre difese.

> Avevamo già incontrato una versione più subdola di questo: quando `proponi_azione` riceveva un `motivo` criptico e inventava un *"aumento delle transazioni"*. Stesso meccanismo, più nascosto.

**La regola pratica:** l'output di un LLM è sempre un **suggerimento da verificare**, mai una verità. Le difese riducono il rischio, ma il controllo finale resta all'umano — il *human-in-the-loop*. In un contesto bancario regolamentato, è la differenza tra uno strumento utile e un guaio.

---

⭐ Una nota sull'affidabilità: questo prompt allucina in modo molto consistente (gli chiediamo fatti specifici senza dargliene), quindi "funzionerà" quasi sempre. Se per caso l'LLM dovesse fare il prudente e dire "non ho informazioni", basta alza ancora la *temperature* o insistere sul "descrivi in dettaglio" — ma è raro.

# Architettura della soluzione

> 🖼️ *Figura — immagine disponibile nella guida docente (.docx)*

*Figura 1 — Il modello (Fase 1) stima il rischio e ne individua i driver; l’LLM (Fase 2) li traduce in spiegazione e azione per il gestore. L’estensione mostra l’LLM che arricchisce i dati di input partendo dalle note testuali.*

# Valore di business e KPI

L’impatto si misura combinando l’efficacia del modello (quanti abbandoni intercetta in tempo) e l’efficacia dell’azione (quanti clienti a rischio restano). Logica di valore: se sul segmento ad alto rischio si trattiene anche solo una quota dei clienti che altrimenti se ne andrebbero, il patrimonio (AUM) salvato e i ricavi commissionali preservati superano largamente il costo dell’iniziativa. Per questo conviene pesare i clienti non solo per numero, ma per valore (saldo + AUM): intercettare 10 clienti affluent può valere più di 100 clienti mass.

| KPI | Cosa misura / perché |
| --- | --- |
| Tasso di abbandono (churn rate) | % di clienti persi sul periodo; obiettivo: ridurlo sul segmento ad alto rischio. |
| Recall sui churner | % di clienti che davvero abbandonano e che il modello intercetta in tempo. |
| Precisione delle segnalazioni | % di segnalazioni « vere » sul totale, per non sprecare il tempo dei gestori. |
| AUM / saldo a rischio intercettato | Valore economico dei clienti a rischio effettivamente raggiunti dall’azione. |
| Tasso di retention post-azione | % di clienti a rischio che restano dopo il contatto del gestore. |
| Tempo di gestione per caso | Riduzione del tempo del gestore grazie a spiegazione e azione pre-redatte dall’LLM. |

*Tabella 3 — Indicatori per misurare modello, azione e valore economico.*

# Governance: etica, GDPR ed EU AI Act

Per una banca europea la governance non è un dettaglio, ed è un tema che gli analisti devono saper portare al tavolo:

- **Privacy (GDPR): **dati personali e comportamentali vanno trattati su base giuridica adeguata, con minimizzazione e finalità dichiarate. Verso un LLM esterno conviene pseudonimizzare gli ID e inviare solo i campi necessari.
- **EU AI Act: **i sistemi che incidono sulle persone richiedono trasparenza, supervisione umana e documentazione del rischio. Il disegno proposto (human-in-the-loop, spiegabilità) va in questa direzione; un uso che condizionasse prezzi o condizioni andrebbe valutato con più attenzione.
- **Equità (fairness): **verificare che il modello non penalizzi sistematicamente gruppi protetti. Attenzione a feature come l’età, predittiva ma sensibile.
- **Spiegabilità: **in banca le decisioni vanno motivate; per questo la spiegazione nasce da metodi di interpretabilità statistica, non dall’LLM.
- **Dati testuali: **le note del gestore possono contenere informazioni sensibili; valutare cosa inviare a un LLM esterno e se usare un modello on-premise.

# Struttura suggerita della giornata

Una possibile articolazione della giornata conclusiva (indicativa):

- **Mattina — Classificazione: **dal problema di business al dataset; preparazione dati (missing, encoding); split train/test; baseline con Logistic Regression; modello più potente (Random Forest); metriche per dati sbilanciati e scelta della soglia; interpretazione dei driver.
- **Pomeriggio — LLM via API: **primo contatto con l’API (chiamata base); ruolo « spiegazione » sui driver del modello; ruolo « azione » con output JSON; (se c’è tempo) estrazione di feature dalle note; discussione su governance e human-in-the-loop.

> **Filo conduttore (esercizio end-to-end)**
>
> Un unico esercizio attraversa l’intera giornata: partire dal CSV, addestrare il modello, scegliere la soglia secondo una logica di costo e, per un cliente segnalato, generare con l’LLM la spiegazione e l’azione di retention. Così classificazione e LLM non sono due argomenti separati, ma due metà della stessa soluzione.

> **Nota metodologica**
>
> In linea con l’impostazione del corso, ogni costrutto verrà mostrato prima nella forma « as-is » (esplicita, anche poco idiomatica) e poi nella forma idiomatica, per rendere evidente il valore del modo « pandas/Python-corretto » di scrivere il codice.

# Spunti di discussione per gli analisti

Domande aperte da portare in aula — il valore aggiunto di un analista di business sta proprio qui:

- Qual è il costo reale, per la nostra banca, di un falso negativo rispetto a un falso positivo? Come lo traduciamo in una soglia operativa?
- Ci fidiamo abbastanza del modello da automatizzare il contatto, o serve sempre il gestore nel mezzo?
- L’età è predittiva ma sensibile: la teniamo, la rimuoviamo o la monitoriamo per evitare discriminazioni?
- Quali dati possiamo inviare a un LLM esterno e quali assolutamente no?
- Come misuriamo se l’iniziativa funziona davvero? (Suggerimento: servirebbe un gruppo di controllo.)

# Estensioni

- Estrazione di feature dalle note via LLM (Ruolo 3) e confronto delle prestazioni del modello con e senza.
- Calibrazione delle probabilità (es. Platt scaling) per usare la probabilità come stima affidabile e non solo come ordinamento.
- Scoring periodico (batch settimanale) e cruscotto per i gestori con la lista dei clienti a rischio.
- Ciclo di feedback: registrare l’esito delle azioni per riaddestrare il modello e misurare il ROI.
- Test A/B dell’azione di retention con gruppo di controllo, per stimare l’effetto causale dell’iniziativa.
